<a href="https://colab.research.google.com/github/rayhan-khan-dev/UIDSS-Optimized-Pipeline/blob/main/UIDSS_Optimized_Pipeline_3rd_edition_rayhan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 0: Install Dependencies
!pip install -q -U sentence-transformers faiss-cpu google-genai pandas numpy
print("Core dependencies installed successfully.")

Core dependencies installed successfully.


In [7]:
# Cell 1: Direct Column Resolution, Aspect & Conflict Tagging, MD5 Deduplication & Local FAISS Indexing
import os
import glob
import hashlib
import json
import re
import numpy as np
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer

# 1. Target Directory Setup
csv_dir = "/content/business_csvs"
os.makedirs(csv_dir, exist_ok=True)

# 2. Aspect Lexicon & Conflict Signals (Research Formulation)
ASPECT_LEXICON = {
    "quality": ["quality", "product", "defect", "durable", "equipment", "machine", "instrument", "মান", "পণ্য"],
    "customer_service": ["service", "staff", "support", "rude", "friendly", "trainer", "coach", "সার্ভিস", "কর্মী", "ব্যবহার"],
    "price_value": ["price", "cost", "value", "expensive", "affordable", "fee", "bill", "দাম", "মূল্য", "খরচ"],
    "cleanliness": ["clean", "dirty", "hygiene", "maintained", "পরিষ্কার", "নোংরা"],
    "speed": ["slow", "fast", "wait", "delay", "late", "quick", "দেরি", "দ্রুত"],
    "environment": ["environment", "ambience", "atmosphere", "space", "crowd", "quiet", "peaceful", "পরিবেশ", "জায়গা"]
}

def extract_aspect_signals(text):
    text_low = str(text).lower()
    mentioned = {}
    for aspect, keywords in ASPECT_LEXICON.items():
        if any(kw in text_low for kw in keywords):
            mentioned[aspect] = True
    return mentioned

def detect_conflict_signal(rating, text):
    text_low = str(text).lower()
    pos_words = ["good", "great", "excellent", "best", "love", "finest", "perfect", "ভালো", "সেরা", "চমৎকার"]
    neg_words = ["bad", "terrible", "awful", "worst", "rude", "poor", "problem", "busy", "gossips", "খারাপ", "বাজে"]

    pos = sum(w in text_low for w in pos_words)
    neg = sum(w in text_low for w in neg_words)

    if rating >= 4.0 and neg > pos:
        return "Hidden Dissatisfaction"
    if rating <= 2.0 and pos > neg:
        return "Politeness Bias"
    return "Consistent"

# 3. Explicit Column Ingestion Function
def ingest_review_file(filepath):
    df = pd.read_csv(filepath)

    # Exact Schema Mapping from Diagnostic Output
    b_col = "business_name"
    t_col = "review_text"
    r_col = "review_rating"

    extracted = pd.DataFrame()
    extracted["business_name"] = df[b_col].fillna("Unknown").astype(str).str.strip()
    extracted["review_text"] = df[t_col].fillna("").astype(str).str.strip()
    extracted["review_rating"] = pd.to_numeric(df[r_col], errors="coerce").fillna(4.0)

    # Clean domain tagging from filename
    fname = os.path.basename(filepath).lower()
    extracted["domain"] = "gym" if "gym" in fname else ("restaurant" if "restaurant" in fname else "general")
    extracted["source_file"] = os.path.basename(filepath)
    return extracted

# 4. Batch Loading
sources = glob.glob(os.path.join(csv_dir, "*.csv"))
if not sources:
    sources = glob.glob("/content/*.csv")

frames = []
for p in sources:
    loaded = ingest_review_file(p)
    frames.append(loaded)
    print(f"Loaded {os.path.basename(p)} -> {len(loaded)} records")

if not frames:
    raise ValueError("No CSV files found in environment.")

merged_df = pd.concat(frames, ignore_index=True)

# 5. Filter Meaningful Text & Vectorized MD5 Deduplication
# Require at least 10 characters of qualitative review text
valid_mask = (merged_df["business_name"].str.len() > 0) & (merged_df["review_text"].str.len() >= 10)
cleaned_df = merged_df[valid_mask].copy().reset_index(drop=True)

def compute_hash(b_name, r_text):
    raw_key = f"{str(b_name).lower().strip()}::{str(r_text).lower().strip()}"
    return hashlib.md5(raw_key.encode("utf-8")).hexdigest()

cleaned_df["content_hash"] = [
    compute_hash(b, t) for b, t in zip(cleaned_df["business_name"], cleaned_df["review_text"])
]
deduped_df = cleaned_df.drop_duplicates(subset=["content_hash"]).reset_index(drop=True)

print(f"\nTotal raw: {len(merged_df)} | Unique qualitative review records: {len(deduped_df)}")

# 6. Build Corpus and Metadata Records
corpus_texts = []
metadata_records = []

for _, row in deduped_df.iterrows():
    aspects = extract_aspect_signals(row["review_text"])
    conflict = detect_conflict_signal(row["review_rating"], row["review_text"])

    content = (
        f"Business: {row['business_name']}\n"
        f"Domain: {row['domain']}\n"
        f"Rating: {row['review_rating']}\n"
        f"Review: {row['review_text']}"
    )
    corpus_texts.append(content)
    metadata_records.append({
        "business_name": str(row["business_name"]),
        "domain": str(row["domain"]),
        "rating": float(row["review_rating"]),
        "source_file": str(row["source_file"]),
        "review_text": str(row["review_text"]),
        "aspects": aspects,
        "conflict_signal": conflict
    })

# 7. Dense Representation & FAISS Vector Indexing
encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
embeddings = encoder.encode(corpus_texts, show_progress_bar=True, normalize_embeddings=True)
embeddings = np.array(embeddings).astype("float32")

dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)
faiss_index.add(embeddings)

print(f"FAISS index built successfully with {faiss_index.ntotal} vectors (dim={dimension}).")

Loaded dhaka_gym.csv -> 197 records
Loaded dhaka_restaurants.csv -> 1013 records

Total raw: 1210 | Unique qualitative review records: 1168


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/37 [00:00<?, ?it/s]

FAISS index built successfully with 1168 vectors (dim=384).


In [8]:
# Cell 2: Resilient Decision Synthesis & Interactive CLI
import time
import getpass
from google import genai
from google.genai import types

# 1. API Client Setup
gemini_api_key = os.environ.get("GEMINI_API_KEY", "").strip()

if not gemini_api_key:
    try:
        from google.colab import userdata
        gemini_api_key = (userdata.get("GEMINI_API_KEY") or "").strip()
    except Exception:
        pass

if not gemini_api_key:
    gemini_api_key = getpass.getpass("Enter Gemini API key: ").strip()

client = genai.Client(api_key=gemini_api_key)
print("Gemini client successfully initialized.")

# 2. Local Evidence Retrieval with Domain Routing
def retrieve_evidence(query, top_k=5, target_domain=None):
    query_vector = encoder.encode([query], normalize_embeddings=True).astype("float32")
    # Fetch wider candidates to support domain filtering
    fetch_k = top_k * 4 if target_domain else top_k
    scores, indices = faiss_index.search(query_vector, fetch_k)

    results = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue
        meta = metadata_records[idx]
        if target_domain and meta["domain"].lower() != target_domain.lower():
            continue

        results.append({
            "score": float(score),
            "business_name": meta["business_name"],
            "domain": meta["domain"],
            "rating": meta["rating"],
            "review_text": meta["review_text"],
            "aspects": meta.get("aspects", {}),
            "conflict": meta.get("conflict_signal", "Consistent")
        })
        if len(results) >= top_k:
            break

    return results

# 3. Degraded Operating Mode (Mode M2 Fallback)
def format_local_evidence_fallback(query, retrieved_items):
    lines = [
        "==================================================",
        "[UIDSS NOTICE: Generation Plane Degraded (Mode M2)]",
        "External LLM generation unavailable (Quota/Rate limit).",
        "Displaying verified local evidence directly from FAISS:",
        "==================================================",
        f"Query: {query}\n"
    ]
    for idx, item in enumerate(retrieved_items, 1):
        aspect_keys = list(item['aspects'].keys())
        aspect_str = ", ".join(aspect_keys) if aspect_keys else "General"
        lines.append(f"Evidence Record [{idx}]")
        lines.append(f"Business: {item['business_name']} ({item['domain'].upper()})")
        lines.append(f"Rating: {item['rating']}/5.0 | Retrieval Score: {item['score']:.4f}")
        lines.append(f"Aspect Signals: {aspect_str} | Conflict Flag: {item['conflict']}")
        lines.append(f"Review Excerpt: {item['review_text']}\n")
    return "\n".join(lines)

# 4. Decision Synthesis Pipeline
def generate_decision_response(query, target_domain=None, top_k=5):
    evidence = retrieve_evidence(query, top_k=top_k, target_domain=target_domain)

    if not evidence:
        return "UIDSS: No relevant review evidence found matching the query."

    context_blocks = []
    for idx, item in enumerate(evidence, 1):
        aspects_detected = list(item['aspects'].keys())
        context_blocks.append(
            f"Evidence Record [{idx}]:\n"
            f"Business: {item['business_name']}\n"
            f"Domain: {item['domain']}\n"
            f"Rating: {item['rating']}\n"
            f"Detected Aspects: {', '.join(aspects_detected) if aspects_detected else 'General'}\n"
            f"Conflict Signal: {item['conflict']}\n"
            f"Customer Review: {item['review_text']}"
        )
    context_text = "\n\n".join(context_blocks)

    prompt = f"""
You are the Universal Intelligent Decision Support System (UIDSS).
Analyze the provided customer review evidence to provide a structured, evidence-grounded recommendation.

OPERATIONAL RULES:
1. Base all statements strictly on the retrieved review evidence below.
2. Do not invent missing facts, specific pricing numbers, or unobserved features.
3. If review text lacks qualitative details, explicitly report this limitation under Section 5.

EVIDENCE CONTEXT:
{context_text}

USER QUERY:
{query}

RESPONSE FORMAT:
1. Executive Summary
2. Key Strengths
3. Weaknesses or Limitations
4. Actionable Recommendation
5. Qualitative Confidence Assessment
6. Grounded Evidence Citations
""".strip()

    max_retries = 3
    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model="gemini-2.5-flash",
                contents=prompt,
                config=types.GenerateContentConfig(
                    temperature=0.2,
                    max_output_tokens=2048
                )
            )
            return response.text.strip()
        except Exception as err:
            if "503" in str(err) and attempt < max_retries - 1:
                time.sleep(3)
                continue
            print(f"\n[Warning] External generation failed: {err}")
            return format_local_evidence_fallback(query, evidence)

# 5. Domain Intent Classifier & CLI Runner
def detect_query_domain(query_text):
    q = query_text.lower()
    gym_keywords = ["gym", "fitness", "workout", "trainer", "exercise", "bodybuilding"]
    restaurant_keywords = ["restaurant", "food", "cafe", "dine", "eat", "taste", "meal"]

    if any(k in q for k in gym_keywords):
        return "gym"
    if any(k in q for k in restaurant_keywords):
        return "restaurant"
    return None

def start_uidss_session():
    print("=" * 60)
    print("UIDSS: Universal Intelligent Decision Support System")
    print("Ready for decision queries (Gyms & Restaurants in Dhaka).")
    print("Type 'exit' or 'quit' to end the session.")
    print("=" * 60 + "\n")

    while True:
        try:
            user_input = input("User Query: ").strip()
            if not user_input:
                continue
            if user_input.lower() in ["exit", "quit", "bye"]:
                print("\nSession terminated. Goodbye!")
                break

            domain = detect_query_domain(user_input)
            response = generate_decision_response(user_input, target_domain=domain, top_k=5)

            print("\n" + "-" * 50)
            print(f"[UIDSS Decision Synthesis] (Target Domain: {domain or 'General'})")
            print("-" * 50)
            print(response)
            print("\n" + "=" * 60 + "\n")

        except KeyboardInterrupt:
            print("\nSession interrupted.")
            break

start_uidss_session()

Enter Gemini API key: ··········
Gemini client successfully initialized.
UIDSS: Universal Intelligent Decision Support System
Ready for decision queries (Gyms & Restaurants in Dhaka).
Type 'exit' or 'quit' to end the session.

User Query: Which gym is best in Dhaka for personal workout and good environment?

[Warning] External generation failed: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 6h46m7.904637453s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.g